# Run your first LiteAgents agent

Get a response with no tools, Temporal, or database setup.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/00_agent.ipynb)

Run these cells in order in **Google Colab**. Everything runs in its cloud runtime:
no repository checkout or laptop installation. A CPU runtime is enough.
Model calls use your provider account. Clear outputs before sharing a saved copy.

## Choose a harness and model

Keep the defaults for a first run. To use another provider, change the model:
- OpenAI: `openai/gpt-5.4-mini` with `OPENAI_API_KEY`.
- Anthropic: `anthropic/claude-sonnet-4-6` with `ANTHROPIC_API_KEY`.
- OpenRouter: `openrouter/anthropic/claude-sonnet-4.6` with `OPENROUTER_API_KEY`.

LiteLLM's Python SDK connects to the provider you choose.
[More providers](https://github.com/BerriAI/liteagents/blob/main/docs/models.md).

In [ ]:
import os

HARNESS = os.environ.get("LITEAGENTS_HARNESS", "deepagents")
MODEL = os.environ.get("LITEAGENTS_MODEL", "openai/gpt-5.4-mini")

Available harnesses: `deepagents`, `pydantic-ai`, `claude-sdk`, `codex`,
`opencode-v1`, `opencode-v2`. Rerun the install cell after changing your selection.
Packages are reused within this runtime; a fresh Colab runtime needs its own install.
OpenCode is installed only when selected.

This preview installs from a GitHub release wheel because the PyPI name currently
belongs to another package. It does not clone the repository.

In [ ]:
# @title Install selected integrations
import shutil
import subprocess
import sys

selected_harnesses = [HARNESS]
extras = sorted(set(selected_harnesses))
release = "https://github.com/BerriAI/liteagents/releases/download/v0.3.0a3"
package = f"liteagents[{','.join(extras)}] @ {release}/liteagents-0.3.0a3-py3-none-any.whl"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", package,
    "-c", f"{release}/constraints-tested.txt",
])
if any(h.startswith("opencode-") for h in selected_harnesses):
    if shutil.which("opencode") is None:
        subprocess.check_call(["npm", "install", "-g", "opencode-ai@1.18.29"])
    subprocess.check_call(["opencode", "--version"])
print("Ready:", ", ".join(selected_harnesses))

## Add your API key

In Colab, open the **key icon → Secrets**, add the key named above, and enable
notebook access. Or enter it in the hidden prompt below. The key stays out of your
code and saved outputs. If installation asks for a runtime restart,
restart once and run the cells again.

In [ ]:
# @title Connect your provider
from getpass import getpass

KEY_NAME = {
    "openai": "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "gemini": "GEMINI_API_KEY",
    "groq": "GROQ_API_KEY",
    "mistral": "MISTRAL_API_KEY",
    "together_ai": "TOGETHERAI_API_KEY",
    "deepseek": "DEEPSEEK_API_KEY",
    "xai": "XAI_API_KEY",
    "azure": "AZURE_API_KEY",
}.get(MODEL.split("/", 1)[0])
API_KEY = None
if KEY_NAME:
    API_KEY = os.environ.get(KEY_NAME)
    if not API_KEY:
        try:
            from google.colab import userdata
        except ImportError:
            pass
        else:
            try:
                API_KEY = userdata.get(KEY_NAME)
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    API_KEY = API_KEY or getpass(f"{KEY_NAME}: ")
    if not API_KEY:
        raise ValueError(f"Provide {KEY_NAME} before running the agent.")
    os.environ[KEY_NAME] = API_KEY
else:
    print("Using provider credentials from the runtime; see the model setup guide.")

## Run your agent

`ProfileOptions` selects the harness and model; `query()` yields LiteAgents messages.
Change `HARNESS` above, rerun the install cell, and run this same code again.
The response interface is modeled after the Claude Agent SDK, regardless of the model provider.

In [ ]:
from liteagents import (
    AssistantMessage,
    LiteAgentOptions,
    ProfileOptions,
    TextBlock,
    query,
)

profile = ProfileOptions(
    harness=HARNESS,
    model=MODEL,
    tools=[],
)
messages = []
async for message in query(
    prompt="Reply with exactly READY.",
    options=LiteAgentOptions(profile=profile),
):
    messages.append(message)
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, TextBlock):
                print(block.text)

## Inspect the messages

Text is in `AssistantMessage.content` as `TextBlock.text`; tool calls use `ToolUseBlock`.
The top-level `query()` starts a fresh conversation each time. For follow-up turns,
see [streaming and conversations](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/01_quickstart.ipynb).

In [ ]:
print("Message types:", [type(message).__name__ for message in messages])
messages

## Next steps

Try [a conversation](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/01_quickstart.ipynb),
[application tools](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/08_application_tools.ipynb),
or [switching harnesses with MCP](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/10_harness_switch.ipynb).

Optional: [gateway setup](https://github.com/BerriAI/liteagents/blob/main/docs/models.md#optional-litellm-gateway)
and [Temporal durability](https://colab.research.google.com/github/BerriAI/liteagents/blob/main/cookbook/recipes/06_durable.ipynb).